# Segment 3 - WARNERCO Schematica & CoALA Memory

**50-minute segment.** The flagship. FastMCP + a 9-node LangGraph RAG pipeline exercising all four CoALA memory tiers (Working / Episodic / Semantic / Procedural). This is the payoff segment.

**What you'll show:**
1. The live primitive census (28 tools / 12 resources / 5 prompts - printed live, never hardcoded)
2. The 9-node pipeline firing end-to-end on a real query, with **Anthropic-generated prose**
3. The four CoALA tiers and where each lives

> Anchor cell first. Pick the **Python 3.13** kernel. The backend `.env` provides the Anthropic key for the reason node.

## Setup: anchor to the repo root

Same backbone. WARNERCO cells shell out with `cwd=WARNERCO` so the backend's relative imports and data paths resolve regardless of where you launched the kernel.

In [1]:
# --- repo-root anchor (identical in all four segment notebooks) ---
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until a directory containing .git is found."""
    for d in [start, *start.parents]:
        if (d / ".git").exists():
            return d
    raise RuntimeError("Repo root (.git) not found above " + str(start))


REPO = find_repo_root(Path.cwd())
WARNERCO = REPO / "src" / "warnerco" / "backend"

print("REPO    :", REPO)
print("WARNERCO:", WARNERCO, "exists:", WARNERCO.exists())
assert WARNERCO.exists(), "WARNERCO backend not found - is REPO correct?"


def warnerco_py(code: str) -> str:
    """Run a Python snippet inside the WARNERCO backend venv + cwd.

    Raises if the snippet errors, so a broken probe fails loudly in validation
    instead of silently printing a traceback as if it were data.
    """
    r = subprocess.run(["uv", "run", "python", "-c", code], cwd=WARNERCO,
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("WARNERCO snippet failed:\n" + (r.stderr or r.stdout))
    noise = ("warning:", "warning", "deprecat", "pydanticdep", "onnxruntime",
             "tqdm", "huggingface", "resume_download", "class ", "dev-depe",
             "dependency-groups")
    return "\n".join(ln for ln in (r.stdout or "").splitlines()
                     if ln.strip() and not any(n in ln.lower() for n in noise))


print("Helper ready: warnerco_py(code)")

REPO    : C:\github\context-engineering
WARNERCO: C:\github\context-engineering\src\warnerco\backend exists: True
Helper ready: warnerco_py(code)


## Live primitive census

The number you quote on stage comes from **here**, not a slide. These counts drift fastest in a teaching repo, so we print them live. WARNERCO uses the standalone `fastmcp` package - `get_tools()` / `get_resources()` / `get_prompts()`.

In [2]:
# Live primitive census - never hardcoded
print(warnerco_py(r'''
import asyncio
from app.mcp_tools import mcp
async def go():
    tools = await mcp.get_tools()
    res = await mcp.get_resources()
    tmpl = await mcp.get_resource_templates()
    prompts = await mcp.get_prompts()
    print(f"TOOLS:     {len(tools)}")
    print(f"RESOURCES: {len(res)} static + {len(tmpl)} template = {len(res)+len(tmpl)} total")
    print(f"PROMPTS:   {len(prompts)}")
    print("Template (the +1):", list(tmpl))
asyncio.run(go())
'''))

TOOLS:     28
RESOURCES: 11 static + 1 template = 12 total
PROMPTS:   5
Template (the +1): ['schematic://{schematic_id}']


## The payoff: run the full 9-node pipeline live

This fires the entire LangGraph RAG pipeline on a real query about a robot that exists in the knowledge base (`Atlas Prime`, model `WC-100`). You'll see all 9 nodes timed, real retrieved results, and **Anthropic-generated prose** from the reason node (node 6).

Pipeline order: `parse_intent -> query_graph -> inject_scratchpad -> recall_episodes -> retrieve -> compress_context -> reason -> respond -> log_episode`.

> First run is ~5-8s (cold imports + one Anthropic call). The reason node uses the `ANTHROPIC_API_KEY` from the backend `.env`.

In [3]:
# Run the full pipeline on a real robot. Shows nodes firing + live Anthropic prose.
print(warnerco_py(r'''
import asyncio
from app.langgraph import run_query
async def go():
    r = await run_query("Tell me about the Atlas Prime (model WC-100) and its capabilities.")
    print("success    :", r.get("success"))
    print("intent     :", r.get("intent"))
    print("results    :", len(r.get("results") or []), "retrieved")
    print("nodes timed:", list((r.get("timings") or {}).keys()))
    rsn = r.get("reasoning", "") or ""
    print("LLM live   :", not rsn.startswith("[LLM unavailable"))
    print("\n--- reason node prose (Anthropic) ---")
    print(rsn[:600])
asyncio.run(go())
'''))

success    : True
intent     : lookup
results    : 5 retrieved
nodes timed: ['parse_intent', 'query_graph', 'inject_scratchpad', 'recall_episodes', 'retrieve', 'compress_context', 'reason', 'log_episode']
LLM live   : True
--- reason node prose (Anthropic) ---
## Atlas Prime (WC-100) — Component Overview
The **Atlas Prime (model WC-100)** is an articulated robot platform with the following documented components:
---
### Active Components
#### 🦾 Precision Gripper Assembly (v2.5) — `WRN-00005`
- **Category:** Manipulation
- **Description:** Adaptive 3-finger gripper with variable force control and tactile feedback, designed for delicate handling operations.
- **Key Specs:**
  - Grip Force: `0.1 – 50N`
  - Finger Count: `3`
  - Positional Accuracy: `0.01mm`
#### 📡 Joint Encoder Module (v4.2) — `WRN-00012`
- **Category:** Sensors
- **Description:**


## The four CoALA memory tiers (Sumers et al. 2024)

| Tier | Storage | Pipeline node | Read tools |
|------|---------|---------------|------------|
| **Working** | `data/scratchpad/notes.db` | `inject_scratchpad` | `warn_scratchpad_read` |
| **Episodic** | `data/episodic/events.db` | `recall_episodes` + `log_episode` | `warn_episodic_recall/recent/stats` |
| **Semantic** | Vector store (Chroma) | `retrieve` | `warn_semantic_search`, `warn_get_robot` |
| **Procedural** | `@mcp.prompt()` registrations | (user-invoked) | `memory://procedural-catalog` |

**Live demo in Claude Code:** `.claude/mcp.json` has two entries - `warnerco-schematica-claude` (general) and `warnerco-coala-memory` (same binary, pinned `EPISODIC_*` weights so recall scores are identical every run). No separate process; the second entry is pedagogical clarity.

**Talk track:** "One query just touched all four tiers. It checked working memory for session notes, recalled past episodes, retrieved from semantic vectors, and logged a new episode on the way out. The prose you see came from Claude, grounded in what the pipeline actually retrieved - not hallucinated."

The cell below proves the consolidation ('sleep cycle') resources are registered.

In [4]:
# Confirm the CoALA-overview resource exists (live four-tier snapshot)
print(warnerco_py(r'''
import asyncio
from app.mcp_tools import mcp
async def go():
    res = await mcp.get_resources()
    coala = [u for u in res if "coala" in u.lower() or "procedural" in u.lower()]
    print("CoALA resources registered:", coala)
asyncio.run(go())
'''))

CoALA resources registered: ['memory://coala-overview', 'memory://procedural-catalog']
